In [ ]:
#Import libraries  
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
#Load dataset  
df = pd.read_csv('/content/spam.csv', encoding='latin-1')


In [ ]:
#Data preprocessing  
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

# Transform label
df['label'] = df['label'].map({'ham': 1, 'spam': 0})

#TF_IDF
vectorizer = TfidfVectorizer(stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
#Train model  
model = LogisticRegression()
model.fit(X_train_vec, y_train)

In [ ]:
#Evaluation  
y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))

In [ ]:
#LIME function
def predict_proba(texts):
    X = vectorizer.transform(texts)
    return model.predict_proba(X)

In [ ]:
#LIME Explanation
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=["spam", "ham"])

exp = explainer.explain_instance(
    X_test.iloc[0],
    predict_proba,
    num_features=6
)

#Result
print(exp.as_list())

In [ ]:
#Visualization
# data from LIME result
result = [
    ('choose', -0.0239),
    ('Funny', 0.0067),
    ('wife', 0.0052),
    ('happens', 0.0037),
    ('hw', 0.0036),
    ('Natural', 0.0033)
]

#Word + weight
words = [w for w, v in result]
values = [v for w, v in result]

In [ ]:
#Bar Chart(Matplotlib)
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.bar(words, values)

plt.title("LIME Explanation")
plt.xlabel("Words")
plt.ylabel("Importance (Weight)")
plt.xticks(rotation=45)

plt.axhline(0, color='black', linewidth=1)
plt.show()

Positive -> Upward
Negative = Downward 

In [ ]:
#Sort from most important to least 
import numpy as np

sorted_idx = np.argsort(values)

words = np.array(words)[sorted_idx]
values = np.array(values)[sorted_idx]

plt.figure(figsize=(8,5))
plt.barh(words, values)

plt.title("LIME Explanation")
plt.xlabel("Importance")
plt.axvline(0, color='black')
plt.show()

1) Negative values (−)
('choose', -0.0239)
means:

This word “pushes the model toward one class” (e.g. ham or spam depending on how the labels are defined)

2) Positive values (+)
('Funny', +0.0067)
('wife', +0.0052)

means:

This word “supports the other class”

3) Why are the values so small?

Because:

LIME uses a local linear approximation
The values are relative weights, not real probabilities

Professional way to read it

When you see something like:

(word, weight)

Think:

weight > 0 → supports the prediction
weight < 0 → opposes the prediction
|weight| → importance of the word